In [2]:
import sqlite3
import pickle

import pandas as pd

In [3]:
conn = sqlite3.connect("../data/raw/database_v2.db")

df = pd.read_sql_query(
    "SELECT * FROM player_stats",
    conn
)

conn.close()

In [7]:
df.shape

(9000, 89)

In [8]:
df["rank_tier"].value_counts().sort_index()

rank_tier
7     600
8     600
9     600
10    600
11    600
12    600
13    600
14    600
15    600
16    600
17    600
18    600
19    600
20    600
21    600
Name: count, dtype: int64

In [9]:
df.isna().sum().sort_values(ascending=False).head(20)

positioning_goals_against_while_last_defender    4070
positioning_percent_closest_to_ball                 1
positioning_percent_most_forward                    1
positioning_time_closest_to_ball                    1
positioning_time_most_forward                       1
positioning_time_most_back                          1
positioning_percent_most_back                       1
rank                                                0
replay_id                                           0
core_saves                                          0
core_assists                                        0
core_score                                          0
core_mvp                                            0
core_shooting_percentage                            0
boost_bpm                                           0
boost_bcpm                                          0
boost_avg_amount                                    0
boost_amount_collected                              0
rank_tier                   

In [11]:
model_df = df.copy()

model_df = model_df.drop(columns=[
    "replay_id",
    "rank",
    "team",
    "duration",
    "positioning_goals_against_while_last_defender"
])

model_df = model_df.dropna()

In [12]:
model_df.shape

(8998, 84)

In [13]:
model_df.isna().sum().sum()

np.int64(0)

In [14]:
X = model_df.drop(columns=["rank_tier"])
y = model_df["rank_tier"]

In [15]:
def tier_to_rank_group(tier):
    if tier <= 9:
        return "gold"
    elif tier <= 12:
        return "platinum"
    elif tier <= 15:
        return "diamond"
    elif tier <= 18:
        return "champion"
    else:
        return "grand_champion"

y_group = y.apply(tier_to_rank_group)

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [17]:
def test_model(model, X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    return model, y_test, y_pred, accuracy

In [18]:
rf_exact_tier_model = RandomForestClassifier(
    random_state=42
)

rf_exact_tier_model, rf_exact_y_test, rf_exact_y_pred, rf_exact_accuracy = test_model(
    rf_exact_tier_model,
    X,
    y
)

rf_exact_accuracy

0.22111111111111112

In [19]:
rf_exact_mae = abs(rf_exact_y_test - rf_exact_y_pred).mean()
rf_exact_within_1 = (abs(rf_exact_y_test - rf_exact_y_pred) <= 1).mean()
rf_exact_within_2 = (abs(rf_exact_y_test - rf_exact_y_pred) <= 2).mean()

rf_exact_mae, rf_exact_within_1, rf_exact_within_2

(np.float64(1.9294444444444445),
 np.float64(0.5),
 np.float64(0.7083333333333334))

In [21]:
rf_broad_rank_model = RandomForestClassifier(
    random_state=42
)

rf_broad_rank_model, rf_broad_y_test, rf_broad_y_pred, rf_broad_accuracy = test_model(
    rf_broad_rank_model,
    X,
    y_group
)

rf_broad_accuracy

0.5222222222222223

In [22]:
log_reg_broad_rank_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42))
])

log_reg_broad_rank_model, log_y_test, log_y_pred, log_accuracy = test_model(
    log_reg_broad_rank_model,
    X,
    y_group
)

log_accuracy

0.5833333333333334

In [23]:
with open("../model/log_reg_broad_rank_model_v2.pkl", "wb") as file:
    pickle.dump(log_reg_broad_rank_model, file)

In [24]:
with open("../model/features_v2.pkl", "wb") as file:
    pickle.dump(list(X.columns), file)

## V2 Model Results

The v2 dataset contains 9000 player rows from 1500 ranked-standard 3v3 replays, with 100 replays per rank tier from Gold 1 through Grand Champion 3.

### Random Forest Exact-Tier Model

The exact-tier Random Forest model predicts one of 15 rank tiers.

Results:

- Exact tier accuracy: 22.1%
- Mean absolute tier error: 1.93 tiers
- Within 1 tier accuracy: 50.0%
- Within 2 tiers accuracy: 70.8%

Exact tier prediction is still difficult, but the model is usually close.

### Broad-Rank Models

The broad-rank models predict one of five rank groups:

- Gold
- Platinum
- Diamond
- Champion
- Grand Champion

Results:

- Random Forest broad-rank accuracy: 52.2%
- Logistic Regression broad-rank accuracy: 58.3%

The Logistic Regression broad-rank model performed best, so it was saved as `log_reg_broad_rank_model_v2.pkl`.

In [28]:
import sqlite3, pandas as pd
conn = sqlite3.connect("../data/raw/database_v2.db")
df = pd.read_sql_query("SELECT * FROM player_stats LIMIT 1", conn)
conn.close()
print(df.columns.tolist())

['replay_id', 'rank', 'rank_tier', 'team', 'duration', 'core_shots', 'core_shots_against', 'core_goals', 'core_goals_against', 'core_saves', 'core_assists', 'core_score', 'core_mvp', 'core_shooting_percentage', 'boost_bpm', 'boost_bcpm', 'boost_avg_amount', 'boost_amount_collected', 'boost_amount_stolen', 'boost_amount_collected_big', 'boost_amount_stolen_big', 'boost_amount_collected_small', 'boost_amount_stolen_small', 'boost_count_collected_big', 'boost_count_stolen_big', 'boost_count_collected_small', 'boost_count_stolen_small', 'boost_amount_overfill', 'boost_amount_overfill_stolen', 'boost_amount_used_while_supersonic', 'boost_time_zero_boost', 'boost_percent_zero_boost', 'boost_time_full_boost', 'boost_percent_full_boost', 'boost_time_boost_0_25', 'boost_time_boost_25_50', 'boost_time_boost_50_75', 'boost_time_boost_75_100', 'boost_percent_boost_0_25', 'boost_percent_boost_25_50', 'boost_percent_boost_50_75', 'boost_percent_boost_75_100', 'movement_avg_speed', 'movement_total_di